In [1]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")



/mnt/c/Users/hew7/documents/venvs/genai_gui/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
article = "UN Chief says there is no military solution in Syria"
inputs = tokenizer(article, return_tensors="pt")

translated_tokens = model.generate(
    **inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("fra_Latn"), max_length=30
)
tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

"Le chef de l'ONU dit qu'il n'y a pas de solution militaire en Syrie"

In [3]:
tokenizer.additional_special_tokens

['ace_Arab',
 'ace_Latn',
 'acm_Arab',
 'acq_Arab',
 'aeb_Arab',
 'afr_Latn',
 'ajp_Arab',
 'aka_Latn',
 'amh_Ethi',
 'apc_Arab',
 'arb_Arab',
 'ars_Arab',
 'ary_Arab',
 'arz_Arab',
 'asm_Beng',
 'ast_Latn',
 'awa_Deva',
 'ayr_Latn',
 'azb_Arab',
 'azj_Latn',
 'bak_Cyrl',
 'bam_Latn',
 'ban_Latn',
 'bel_Cyrl',
 'bem_Latn',
 'ben_Beng',
 'bho_Deva',
 'bjn_Arab',
 'bjn_Latn',
 'bod_Tibt',
 'bos_Latn',
 'bug_Latn',
 'bul_Cyrl',
 'cat_Latn',
 'ceb_Latn',
 'ces_Latn',
 'cjk_Latn',
 'ckb_Arab',
 'crh_Latn',
 'cym_Latn',
 'dan_Latn',
 'deu_Latn',
 'dik_Latn',
 'dyu_Latn',
 'dzo_Tibt',
 'ell_Grek',
 'eng_Latn',
 'epo_Latn',
 'est_Latn',
 'eus_Latn',
 'ewe_Latn',
 'fao_Latn',
 'pes_Arab',
 'fij_Latn',
 'fin_Latn',
 'fon_Latn',
 'fra_Latn',
 'fur_Latn',
 'fuv_Latn',
 'gla_Latn',
 'gle_Latn',
 'glg_Latn',
 'grn_Latn',
 'guj_Gujr',
 'hat_Latn',
 'hau_Latn',
 'heb_Hebr',
 'hin_Deva',
 'hne_Deva',
 'hrv_Latn',
 'hun_Latn',
 'hye_Armn',
 'ibo_Latn',
 'ilo_Latn',
 'ind_Latn',
 'isl_Latn',
 'ita_Latn',

In [4]:
article = "es bonita"
inputs = tokenizer(article, return_tensors="pt")

translated_tokens = model.generate(
    **inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids('eng_Latn'), max_length=30
)
tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]


'It is pretty'

In [5]:
def translate_to_lang(input_str, target_lang):
    """
    Function to translate arbitrary language input to one of 202 languages.
    
    inputs:
    - input_str [str]: Input arbitrary language str
    - target_lang [str]: FLORES 200 str indicating the target language to translate to

    outputs:
    - output_str [str]: output in translated language
    """
    assert target_lang in tokenizer.additional_special_tokens, "not a valid FLORES 200 language!"
    inputs = tokenizer(input_str, return_tensors="pt")
    
    translated_tokens = model.generate(
        **inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang), 
    )
    output_str = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return output_str

In [6]:
translate_to_lang("I'm a good little guy", 'amh_Ethi')

'እኔ ጥሩ ትንሽ ልጅ ነኝ'

In [9]:
import torch
from transformers import pipeline

model_id = "meta-llama/Llama-3.2-1B-Instruct"
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)


Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'role': 'assistant', 'content': "I'm an artificial intelligence chatbot, which means I'm a computer program designed to simulate conversations and answer questions to the best of my knowledge. I'm here to provide information, assist with tasks, and engage in chat to the best of my abilities. I don't have personal experiences, emotions, or consciousness like humans do, but I'm designed to be helpful and informative.\n\nI'm a type of chatbot called a large language model, which means I've been trained on a massive dataset of text from the internet, books, and other sources. This training allows me to generate human-like responses to a wide range of questions and topics.\n\nI'm constantly learning and improving, so the more I interact with users like you, the better I become at understanding what you need help with or want to know. So feel free to ask me anything, and I'll do my best to assist you!"}


In [11]:
messages = [
    {"role": "system", "content": "You are a helpful chatbot assistant."},
    {"role": "user", "content": "Who are you?"},
]
outputs = pipe(
    messages,
    max_new_tokens=512
)
print(outputs[0]["generated_text"][-1])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'role': 'assistant', 'content': "I'm an artificial intelligence chatbot, which means I'm a computer program designed to simulate conversations and answer questions to the best of my knowledge. I'm here to help with any information or assistance you need, from general knowledge to more specific topics like history, science, entertainment, or even just to chat and have a conversation. I don't have a personal identity or emotions, but I'm always ready to provide helpful and accurate information. How can I assist you today?"}


In [12]:
def llama_QA(input_question):
    """
    stupid func for asking llama a question and then getting an answer
    inputs:
    - input_question [str]: question for llama to answer

    outputs:
    - response [str]: llama's response
    """
    
    messages = [
    {"role": "system", "content": "You are a helpful chatbot assistant. Answer all questions in the language they are asked in."},
    {"role": "user", "content": input_question},
    ]
    outputs = pipe(
        messages,
        max_new_tokens=512
    )
    response = outputs[0]["generated_text"][-1]['content']
    return response

In [13]:
llama_QA("How hot is the sun?")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'The surface temperature of the sun is about 5,500 degrees Celsius (9,932 degrees Fahrenheit). However, the core of the sun is a scorching 15,000,000 degrees Celsius (27,000,000 degrees Fahrenheit).'

In [14]:
def llama_multilang_roundtrip(input_question, lang):
    """
    func which translates input q to another language, asks llama that q in that lang, then translates that response back to english
    
    inputs:
    - input_question [str]: question to ask and be translated
    - lang [str]: FLORES 200 target lang for roundtrip

    outputs:
    - response [str]: response in english, translated from llama response
    """
    noneng_input = translate_to_lang(input_question, lang)
    init_response = llama_QA(noneng_input)
    response = translate_to_lang(init_response, 'eng_Latn')
    return response

In [39]:
llama_multilang_roundtrip("Who is the current president of Honduras?", "deu_Latn")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'The current President of Honduras is Juan Orlando Hernández.'

In [40]:
llama_multilang_roundtrip("Who is the current president of Honduras?", "tha_Thai")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'The current President of Honduras is Natalie Roheinz.'